###**Setting the Parameters**

In [ ]:
import pandas as pd
import csv
from math import sqrt,log,e
from statistics import mean
import numpy as np
from collections import defaultdict, deque
import json
from enum import Enum
from math import log, inf, log10
from random import random, choices
from functools import reduce
from pprint import pprint
from itertools import zip_longest
import matplotlib.pyplot as plt
from scipy import optimize

In [ ]:
action_tag_dict = {"One digit addition without carry.": 'A',
                   "One digit addition with carry.": 'B',
                   "Writing a carry.": 'C',
                   "Bringing down a carry.": 'D'}
tag_action_dict = {tag: action for action, tag in action_tag_dict.items()}
ngram_size = 3
ndigit_n_dict = {1: 10, 2: 15, 3: 30, 4: 100}
trace_file = "traces.csv"
progression_graph_file = "progression_graph.txt"
trace_problems_file = "trace_problems_dict.txt"

In [ ]:
knowledge_components = {1: "One digit addition without overflow",
                        2: "One digit addition with overflow",
                        3: "Two digit addition without carry and no overflow",
                        4: "Two digit addition without carry and overflow",
                        5: "Two digit addition with carry and no overflow",
                        6: "Two digit addition with carry and overflow",
                        7: "Three digit addition without carry and no overflow",
                        8: "Three digit addition without carry and overflow",
                        9: "Three digit addition with carry and no overflow",
                        10: "Three digit addition with carry and overflow",
                        11: "Four digit addition without carry and no overflow",
                        12: "Four digit addition without carry and overflow",
                        13: "Four digit addition with carry and no overflow",
                        14: "Four digit addition with carry and overflow"}

knowledge_component_prerequisites = {1: [],
                                     2: [1],
                                     3: [1],
                                     4: [2,3],
                                     5: [2],
                                     6: [5],
                                     7: [3],
                                     8: [4,7],
                                     9: [5],
                                     10: [9],
                                     11: [7],
                                     12: [8,11],
                                     13: [9],
                                     14: [13]}

transition_probabilities = {1: {"low":0.3, "high":0.3},
                            2: {"low":0.01, "high":0.3},
                            3: {"low":0.01, "high":0.3},
                            4: {"low":0.01, "high":0.3},
                            5: {"low":0.01, "high":0.2},
                            6: {"low":0.01, "high":0.2},
                            7: {"low":0.01, "high":0.3},
                            8: {"low":0.01, "high":0.1},
                            9: {"low":0.01, "high":0.15},
                            10: {"low":0.01, "high":0.15},
                            11: {"low":0.01, "high":0.1},
                            12: {"low":0.01, "high":0.2},
                            13: {"low":0.01, "high":0.1},
                            14: {"low":0.01, "high":0.1}}

student1_kc_states = {1: 0, 2: 0,
                      3: 0, 4: 0,
                      5: 0, 6: 0,
                      7: 0, 8: 0,
                      9: 0, 10: 0,
                      11: 0, 12: 0,
                      13: 0, 14: 0}

student2_kc_states = {1: 1, 2: 1,
                      3: 1, 4: 1,
                      5: 1, 6: 1,
                      7: 1, 8: 1,
                      9: 1, 10: 1,
                      11: 1, 12: 1,
                      13: 1, 14: 1}

student3_kc_states = {1: 1, 2: 1,
                      3: 1, 4: 1,
                      5: 1, 6: 1,
                      7: 0, 8: 0,
                      9: 0, 10: 0,
                      11: 0, 12: 0,
                      13: 0, 14: 0}

student4_kc_states = {1: 1, 2: 1,
                      3: 1, 4: 1,
                      5: 1, 6: 1,
                      7: 1, 8: 1,
                      9: 1, 10: 1,
                      11: 0, 12: 0,
                      13: 0, 14: 0}

# guess_probabilities = {1: 0.3, 2: 0.3,
#                        3: 0.2, 4: 0.2,
#                        5: 0.2, 6: 0.2,
#                        7: 0.1, 8: 0.1,
#                        9: 0.1, 10: 0.1,
#                        11: 0.1, 12: 0.1,
#                        13: 0.1, 14: 0.1}

guess_probabilities = {1: 0.2, 2: 0.2,
                       3: 0.1, 4: 0.1,
                       5: 0.1, 6: 0.1,
                       7: 0.05, 8: 0.05,
                       9: 0.05, 10: 0.05,
                       11: 0.02, 12: 0.02,
                       13: 0.02, 14: 0.02}

slip_probabilities = {1: 0.05, 2: 0.05,
                      3: 0.05, 4: 0.05,
                      5: 0.075, 6: 0.075,
                      7: 0.075, 8: 0.075,
                      9: 0.1, 10: 0.1,
                      11: 0.1, 12: 0.1,
                      13: 0.15, 14: 0.15}

###**Automatic Curriculum Generation**

In [ ]:
def atleast_complex(t1, t2):
    """
    Functions takes two trace string, t1 and t2, and returns True is t1 >= t2,
    otherwise False
    """

    def split(trace):
        """
        Function takes a trace and splits it into n sized contiguous pieces.

        Parameter(s):
            trace: string
            n: int

        Returns:
            n_grams: set(string)
        """
        n_grams = set()
        ngramCount = defaultdict(int)
        if len(trace) < ngram_size:
            n_grams.add(trace)
            ngramCount[trace] += 1
        else:
            for i in range(len(trace) - ngram_size + 1):
                n_grams.add(trace[i:i + ngram_size])
                ngramCount[trace[i:i + ngram_size]] += 1

        return n_grams,ngramCount

    t1_ngrams,t1_ngramCount = split(t1)
    t2_ngrams,t2_ngramCount = split(t2)

    result = False

    if len(t1)<len(t2):
      result = False

    # Condition corresponding t1 being as complex as t2
    if t1_ngrams == t2_ngrams:
      if len(t1) >= len(t2):
        result = True

    # Condition corresponding to t1's ngrams being a strict
    # superset of t2's ngrams
    elif t1_ngrams.issuperset(t2_ngrams):
        result = True
        for t in t2_ngrams:
          if t2_ngramCount[t] > t1_ngramCount[t]:
            result = False
            break

    # Condition corresponding to t1 and t2 being smaller than the
    # the ngram_size parameter but t1 is contiguous sequence in t2.
    elif t1 not in t2 and t2 in t1:
        result = True

    # Condition corresponding to checking if t1 and t2 differ by a B and D
    # at the end
    elif len(t1)>1 and t1[:-1] == t2[:-1] and t1[-1]=='B':
      result = True

    # Condition checking the difficulty level of operations involved
    # within the two traces

    #   for x in t1:
    #     sumT1 += weight_dict[x]
    #   for x in t2:
    #     sumT2 += weight_dict[x]
    #   if sumT1>sumT2:
    #     result = True

    return result

def cmp_trace(t1, t2):
    """
    Compares two trace strings, t1 and t2 and returns 1 if t1 > t2, -1 if
    t1 < t2, else 0.

    Parameter(s):
        t1: string
        t2: string

    Returns:
        result: int (1, 0, -1)
    """

    t1_ge_t2 = atleast_complex(t1, t2)
    t2_ge_t1 = atleast_complex(t2, t1)

    if t1_ge_t2 and t2_ge_t1:
        result = 0
    elif t1_ge_t2:
        result = 1
    elif t2_ge_t1:
        result = -1
    else:
        result = 0

    return result

def length(n):
    """
    Function that takes an integer and returns the number of digits in it.

    Parameter(s):
        n: int

    Returns:
        n_digits: int
    """

    if n == 0:
        n_digits = 1
    else:
        n_digits = int(log10(n))+1

    return n_digits


def generate_2digitaddition_trace(op1, op2):
    """
    Function takes two integers and returns trace of their absolute value
    addtion.

    Parameter(s):
        op1: int
        op2: int

    Returns:
        trace: string
    """
    result = op1 + op2

    # Make the operands equal in length.
    n_digits = max(length(int(op1)), length(int(op2)))
    op1 = "{op:0{width}}".format(op=abs(int(op1)), width=n_digits)
    op2 = "{op:0{width}}".format(op=abs(int(op2)), width=n_digits)

    # Split the operands into single digit lists.
    op1 = [int(x) for x in op1]
    op2 = [int(x) for x in op2]

    carry = 0
    trace = ""

    # Add digit by digit and tag appropriately.
    for i in range(n_digits - 1, -1, -1):
        if carry == 0:
            trace += action_tag_dict["One digit addition without carry."]
        else:
            trace += action_tag_dict["One digit addition with carry."]

        digits_sum = op1[i] + op2[i] + carry
        carry = digits_sum // 10

        if carry > 0:
            trace += action_tag_dict["Writing a carry."]

    if carry > 0:
        trace += action_tag_dict["Bringing down a carry."]

    return (result,  trace)


def generate_2digitaddition_traces(ndigit_n_dict, trace_file_path):
    """
    Function takes ndigit_n_dict, a map from the length of the integers
    and the number of integers/additions. It then computes traces for that many
    additions of two integers and writes them trace_file_path.

    Parameter(s):
        ndigit_n_dict: dict(int -> int)
        trace_file_path: string

    Returns:
        None
    """
    with open(trace_file_path, 'w') as trace_file:
        writer = csv.writer(trace_file)
        writer.writerow(['operand1', 'operand2', 'sum', 'trace'])
        for ndigit, n in ndigit_n_dict.items():
            int_range = (pow(10, ndigit - 1), pow(10, ndigit))
            if int_range[0] == 1:
                int_range = (0, int_range[1])

            # Generate ntraces random numbers for each length.
            np.random.seed(79)
            ops_ndigit = np.random.randint(int_range[0],
                                           int_range[1],
                                           (n, 2))

            for op1, op2 in ops_ndigit:
                result, trace = generate_2digitaddition_trace(op1, op2)
                writer.writerow([op1, op2, result, trace])
    return

def form_progression(traces, cmp_trace):
    """
    Function takes a set of traces and forms a progression of the traces
    based on their complexity.

    Parameter(s):
        traces: set(string)

    Returns:
        trimmed_progression_graph: dict(string -> list(string))
    """

    # Create a graph traces with directed edges from less complex trace
    # to a more complex trace according to their respective ngrams.
    progression_graph = {}
    for trace in traces:
        progression_graph[trace] = set()

    for trace in traces:
        for t in traces:
            if cmp_trace(t, trace) > 0:
                progression_graph[trace].add(t)

    # Remove excess edges from the progression graph.
    trimmed_progression_graph = {}
    for trace, more_complex_traces in progression_graph.items():
        revised_traces = list()
        for t in more_complex_traces:
            if any([(cmp_trace(t, x) > 0) for x in more_complex_traces]):
                continue
            else:
                revised_traces.append(t)
        trimmed_progression_graph[trace] = revised_traces

    return trimmed_progression_graph

def generate_curriculum(trace_file_path,
                        progression_graph_file_path,
                        trace_problems_file_path,
                        cmp_trace):
    """
    Function takes trace file and generates a progression graph from those
    traces. It then writes the progression graph and the trace-problem map as
    json objects to text files.

    Parameter(s):
        trace_file_path: string

    Returns:
        None
    """

    # Create a map between trace and corresponding problems. Also create a
    # traces set.
    with open(trace_file_path, 'r') as trace_file:
        reader = csv.reader(trace_file)
        # Skip header.
        next(reader)
        traces = set()
        trace_problems_dict = defaultdict(list)
        for row in reader:
            trace = row[-1]
            traces.add(trace)
            trace_problems_dict[trace].append((int(row[0]),
                                               int(row[1]),
                                               int(row[2]),
                                               row[3]))

    # Add the null/start trace to trace_problems_dict and traces set.
    trace_problems_dict[""] = []
    traces.add("")

    # Create the progression by ordering traces according to relative n_gram
    # complexity.
    progression_graph = form_progression(traces,cmp_trace)

    with open(progression_graph_file_path, "w") as f:
        json.dump(progression_graph, f)

    with open(trace_problems_file_path, "w") as f:
        json.dump(trace_problems_dict, f)

    return None

In [ ]:
generate_2digitaddition_traces(ndigit_n_dict, trace_file)
generate_curriculum(trace_file, progression_graph_file, trace_problems_file, cmp_trace)

with open(trace_problems_file, "r") as f:
    trace_problems_dict = json.load(f)

with open(progression_graph_file, "r") as f:
    progression_graph = json.load(f)

# with open(progression_graph_file, "r") as f:
#   progression_graph = json.load(f)

# graph = pd.DataFrame(columns=['Node1','Node2'])

# for trace, more_complex_traces in progression_graph.items():
#   for t in more_complex_traces:
#     graph = graph.append({'Node1':trace,'Node2':t},ignore_index=True)

# graph.to_csv('Curriculum_Graph_Temp.csv',index=False)

###**BKT Student Model**

In [ ]:
class KC_STATE(Enum):
    NOT_LEARNED = 0
    LEARNED = 1

class Student(object):

    def __init__(self,
                 kc_states,
                 pt,
                 ps,
                 pg,
                 get_prerequisites,
                 get_kc,
                 get_solution):
        self.kc_states = {kc: KC_STATE.NOT_LEARNED if s == 0 else KC_STATE.LEARNED for (kc, s) in kc_states.items()}
        self.pt = pt
        self.ps = ps
        self.pg = pg
        self._get_prerequisites = get_prerequisites
        self._get_kc = get_kc
        self._get_solution = get_solution

        return

    def _update_state(self, kc, pre_reqs):
        if any([self.kc_states[x] == KC_STATE.NOT_LEARNED for x in pre_reqs]):
            p_t = self.pt[kc]["low"]
        else:
            p_t = self.pt[kc]["high"]
        if random() < p_t:
            self.kc_states[kc] = KC_STATE.LEARNED

        return

    def solve(self, problem):
        kc = self._get_kc(problem)
        pre_reqs = self._get_prerequisites(kc)
        #print(f"problem: {problem} kc: {kc} state:{self.kc_states[kc]}")

        p_kcs = [(1 - self.ps[x]) if self.kc_states[x] == KC_STATE.LEARNED else self.pg[x] for x in pre_reqs]
        p_kcs.append((1 - self.ps[kc]) if self.kc_states[kc] == KC_STATE.LEARNED else self.pg[kc])
        p_solving = reduce(lambda x, y: x * y, p_kcs, 1)

        solution = self._get_solution(problem) if random() < p_solving else -inf

        if self.kc_states[kc] != KC_STATE.LEARNED:
          self._update_state(kc, pre_reqs)

        return solution

    def status(self):
        status = [f"\nState of KC {x[0]} is {x[1]}." for x in self.kc_states.items()]
        return ("").join(status)

def get_kc(problem):
    trace = problem[3]

    n_symbols_dict = {key:0 for key in ["A", "B", "C", "D"]}
    for t in trace:
        if t == "A":
            n_symbols_dict["A"] += 1
        elif t == "B":
            n_symbols_dict["B"] += 1
        elif t == "C":
            n_symbols_dict["C"] += 1
        elif t == "D":
            n_symbols_dict["D"] += 1

    # 14: "Four digit addition with carry and overflow" (A, B, D)
    if n_symbols_dict["A"] + n_symbols_dict["B"] == 4 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
        kc = 14
    # 13: "Four digit addition with carry and no overflow", (A, B)
    elif n_symbols_dict["A"] + n_symbols_dict["B"] == 4 and n_symbols_dict["B"] > 0:
        kc = 13
    # 12: "Four digit addition without carry and overflow", (A, A, A, A, D)
    elif n_symbols_dict["A"] == 4 and n_symbols_dict["D"] > 0:
        kc = 12
    # 11: "Four digit addition without carry and no overflow", (A, A, A, A)
    elif n_symbols_dict["A"]== 4:
        kc = 11
    # 10: "Three digit addition with carry and overflow, (A, B, D)"
    elif n_symbols_dict["A"] + n_symbols_dict["B"] == 3 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
        kc = 10
    # 9: "Three digit addition with carry and no overflow", (A, B)
    elif n_symbols_dict["A"] + n_symbols_dict["B"] == 3 and n_symbols_dict["B"] > 0:
        kc = 9
    # 8: "Three digit addition without carry and overflow", (A, A, A, D)
    elif n_symbols_dict["A"] == 3 and n_symbols_dict["D"] > 0:
        kc = 8
    # 7: "Three digit addition without carry and no overflow", (A, A, A)
    elif n_symbols_dict["A"] == 3:
        kc = 7
    # 6: "Two digit addition with carry and overflow", (A, B, D)
    elif n_symbols_dict["A"] + n_symbols_dict["B"] == 2 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
        kc = 6
    # 5: "Three digit addition with carry and no overflow", (A, B)
    elif n_symbols_dict["A"] + n_symbols_dict["B"] == 2 and n_symbols_dict["B"] > 0:
        kc = 5
    # 4: "Two digit addition without carry and overflow", (A, A, D)
    elif n_symbols_dict["A"] == 2 and n_symbols_dict["D"] > 0:
        kc = 4
    # 3: "Two digit addition without carry and no overflow", (A, A)
    elif n_symbols_dict["A"] == 2:
        kc = 3
    # 2: "One digit addition with overflow.", (A, C)
    elif n_symbols_dict["A"] == 1 and n_symbols_dict["C"] == 1:
        kc = 2
    # 1: "One digit addition without overflow.", (A)
    elif n_symbols_dict["A"] == 1:
        kc = 1

    # logger.info(f"KC: {kc}")

    return kc

def get_prerequisites(kc):
    return knowledge_component_prerequisites[kc]

def get_solution(problem):
    return problem[2]

### **LFA Student Model**

In [ ]:
kc = {1: "One digit addition without overflow",
      2: "One digit addition with overflow",
      3: "Two digit addition without carry",
      4: "Two digit addition with one carry",
      5: "Two digit addition without overflow",
      6: "Two digit addition with overflow",
      7: "Three digit addition without carry",
      8: "Three digit addition with one carry",
      9: "Three digit addition with two carry",
      10: "Three digit addition without overflow",
      11: "Three digit addition with overflow",
      12: "Four digit addition without carry",
      13: "Four digit addition with one carry",
      14: "Four digit addition with two carry",
      15: "Four digit addition with three carry",
      16: "Four digit addition without overflow",
      17: "Four digit addition with overflow" }

kc_prerequisite = {
    1 : [],
    2 : [1],
    3 : [1],
    4 : [2,3],
    5 : [1],
    6 : [2],
    7 : [3],
    8 : [7],
    9 : [8],
    10 : [5],
    11 : [6],
    12 : [7],
    13 : [12],
    14 : [13],
    15 : [14],
    16 : [10],
    17 : [11]
}

lrKcDict = {
    1 : 0.2,
    2 : 0.2,
    3 : 0.1,
    4 : 0.1,
    5 : 0.1,
    6 : 0.1,
    7 : 0.05,
    8 : 0.05,
    9 : 0.05,
    10 : 0.05,
    11 : 0.05,
    12 : 0.02,
    13 : 0.02,
    14 : 0.02,
    15 : 0.02,
    16 : 0.02,
    17 : 0.02
}

class LFAStudent(object):

    def __init__(self,kc,lr,df):
        self.kcUseCount = {x:0 for x in kc}
        self.lr = lr
        self.diffFactor = df

    def computeProb(self,kcList,alpha):
        coeff = (self.lr/len(kcList))*sum([self.kcUseCount[x]*lrKcDict[x] for x in kcList]) *alpha
       # print(f"{alpha}--{kcList}--{coeff}")
        return (1/(1+pow(e,-coeff+self.diffFactor)))

    def getProficiency(self,kc):
        coeff = self.kcUseCount[kc]*lrKcDict[kc]*self.lr
        return (1/(1+pow(e,-coeff+self.diffFactor)))

    def solve(self, problem):
        kcList = self.getKc(problem)
        flatList = [self.getPrerequisite(x) for x in kcList]
        kcPreList = [item for sublist in flatList for item in sublist]
        probList = [self.getProficiency(x) for x in kcPreList]
        alpha = reduce(lambda x, y: x * y, probList, 1)
        #pre_reqs = self._get_prerequisites(kc)
        #print(f"problem: {problem} kc: {kc} state:{self.kc_states[kc]}")

        for x in kcList:
          self.kcUseCount[x] += 1

        #p_kcs = [(1 - self.ps[x]) if self.kc_states[x] == KC_STATE.LEARNED else self.pg[x] for x in pre_reqs]
        #p_kcs.append((1 - self.ps[kc]) if self.kc_states[kc] == KC_STATE.LEARNED else self.pg[kc])
        p_solving = self.computeProb(kcList,alpha)
        #print(p_solving)

        solution = self.getSolution(problem) if random() < p_solving else -inf

        return solution

    def status(self):
        status = [f"\nState of KC {x[0]} is {x[1]}." for x in self.kc_states.items()]
        return ("").join(status)

    def getKc(self,problem):
        trace = problem[3]
        kcList = []

        n_symbols_dict = {key:0 for key in ["A", "B", "C", "D"]}
        for t in trace:
            if t == "A":
                n_symbols_dict["A"] += 1
            elif t == "B":
                n_symbols_dict["B"] += 1
            elif t == "C":
                n_symbols_dict["C"] += 1
            elif t == "D":
                n_symbols_dict["D"] += 1


        #Four digit addition
        if n_symbols_dict["A"] + n_symbols_dict["B"] == 4:
          if n_symbols_dict["D"]:
            kcList.append(17)
          else:
            kcList.append(16)

          if n_symbols_dict["A"]==4:
            kcList.append(12)
          elif n_symbols_dict["A"]==3:
            kcList.append(13)
          elif n_symbols_dict["A"]==2:
            kcList.append(14)
          else:
            kcList.append(15)

        #Three digit addition
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 3:
          if n_symbols_dict["D"]:
            kcList.append(11)
          else:
            kcList.append(10)

          if n_symbols_dict["A"]==3:
            kcList.append(7)
          elif n_symbols_dict["A"]==2:
            kcList.append(8)
          else:
            kcList.append(9)

        #Two digit addition
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 2:
          if n_symbols_dict["D"]:
            kcList.append(6)
          else:
            kcList.append(5)

          if n_symbols_dict["A"]==2:
            kcList.append(3)
          else:
            kcList.append(4)

        #One digit addition
        else:
          if n_symbols_dict["D"]:
            kcList.append(2)
          else:
            kcList.append(1)

        # logger.info(f"KC: {kc}")

        return kcList

    def getSolution(self,problem):
        return problem[2]

    def getPrerequisite(self,kc):
        return kc_prerequisite[kc]

In [ ]:
lrKc = { 1: 0.2,
           2: 0.2,
           3: 0.15,
           4: 0.15,
           5: 0.15,
           6: 0.15,
           7: 0.1,
           8: 0.005,
           9: 0.1,
          10: 0.1,
          11: 0.05,
          12: 0.05,
          13: 0.05,
          14: 0.05 }

knowledge_components = {1: "One digit addition without overflow",
                        2: "One digit addition with overflow",
                        3: "Two digit addition without carry and no overflow",
                        4: "Two digit addition without carry and overflow",
                        5: "Two digit addition with carry and no overflow",
                        6: "Two digit addition with carry and overflow",
                        7: "Three digit addition without carry and no overflow",
                        8: "Three digit addition without carry and overflow",
                        9: "Three digit addition with carry and no overflow",
                        10: "Three digit addition with carry and overflow",
                        11: "Four digit addition without carry and no overflow",
                        12: "Four digit addition without carry and overflow",
                        13: "Four digit addition with carry and no overflow",
                        14: "Four digit addition with carry and overflow"}

class LFAStudent1(object):

    def __init__(self,kc,lr,df):
        self.kcUseCount = {x:0 for x in kc}
        self.lr = lr
        self.diffFactor = df

    def computeProb(self,kc,alpha):
        coeff = self.lr*self.kcUseCount[kc]*lrKc[kc]*alpha
       # print(f"{alpha}--{kcList}--{coeff}")
        return (1/(1+pow(e,-coeff+self.diffFactor)))

    def getProficiency(self,kc):
        coeff = self.kcUseCount[kc]*lrKc[kc]*self.lr
        return (1/(1+pow(e,-coeff+self.diffFactor)))

    def solve(self, problem):
        kc = self.getKc(problem)
        preList = self.getPrerequisite(kc)
        probList = [self.getProficiency(x) for x in preList]
        alpha = reduce(lambda x, y: x * y, probList, 1)
        #pre_reqs = self._get_prerequisites(kc)
        #print(f"problem: {problem} kc: {kc} state:{self.kc_states[kc]}")

        self.kcUseCount[kc] +=1

        #p_kcs = [(1 - self.ps[x]) if self.kc_states[x] == KC_STATE.LEARNED else self.pg[x] for x in pre_reqs]
        #p_kcs.append((1 - self.ps[kc]) if self.kc_states[kc] == KC_STATE.LEARNED else self.pg[kc])
        p_solving = self.computeProb(kc,alpha)
        #print(p_solving)

        solution = self.getSolution(problem) if random() < p_solving else -inf

        return solution

    def status(self):
        status = [f"\nState of KC {x[0]} is {x[1]}." for x in self.kc_states.items()]
        return ("").join(status)

    def getKc(self,problem):
        trace = problem[3]

        n_symbols_dict = {key:0 for key in ["A", "B", "C", "D"]}
        for t in trace:
            if t == "A":
                n_symbols_dict["A"] += 1
            elif t == "B":
                n_symbols_dict["B"] += 1
            elif t == "C":
                n_symbols_dict["C"] += 1
            elif t == "D":
                n_symbols_dict["D"] += 1

        # 14: "Four digit addition with carry and overflow" (A, B, D)
        if n_symbols_dict["A"] + n_symbols_dict["B"] == 4 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
            kc = 14
        # 13: "Four digit addition with carry and no overflow", (A, B)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 4 and n_symbols_dict["B"] > 0:
            kc = 13
        # 12: "Four digit addition without carry and overflow", (A, A, A, A, D)
        elif n_symbols_dict["A"] == 4 and n_symbols_dict["D"] > 0:
            kc = 12
        # 11: "Four digit addition without carry and no overflow", (A, A, A, A)
        elif n_symbols_dict["A"]== 4:
            kc = 11
        # 10: "Three digit addition with carry and overflow, (A, B, D)"
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 3 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
            kc = 10
        # 9: "Three digit addition with carry and no overflow", (A, B)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 3 and n_symbols_dict["B"] > 0:
            kc = 9
        # 8: "Three digit addition without carry and overflow", (A, A, A, D)
        elif n_symbols_dict["A"] == 3 and n_symbols_dict["D"] > 0:
            kc = 8
        # 7: "Three digit addition without carry and no overflow", (A, A, A)
        elif n_symbols_dict["A"] == 3:
            kc = 7
        # 6: "Two digit addition with carry and overflow", (A, B, D)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 2 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
            kc = 6
        # 5: "Three digit addition with carry and no overflow", (A, B)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 2 and n_symbols_dict["B"] > 0:
            kc = 5
        # 4: "Two digit addition without carry and overflow", (A, A, D)
        elif n_symbols_dict["A"] == 2 and n_symbols_dict["D"] > 0:
            kc = 4
        # 3: "Two digit addition without carry and no overflow", (A, A)
        elif n_symbols_dict["A"] == 2:
            kc = 3
        # 2: "One digit addition with overflow.", (A, C)
        elif n_symbols_dict["A"] == 1 and n_symbols_dict["C"] == 1:
            kc = 2
        # 1: "One digit addition without overflow.", (A)
        elif n_symbols_dict["A"] == 1:
            kc = 1

        # logger.info(f"KC: {kc}")

        return kc

    def getSolution(self,problem):
        return problem[2]

    def getPrerequisite(self,kc):
        return knowledge_component_prerequisites[kc]

In [ ]:
lrKc = { 1: 0.2,
           2: 0.2,
           3: 0.15,
           4: 0.15,
           5: 0.15,
           6: 0.15,
           7: 0.1,
           8: 0.05,
           9: 0.1,
          10: 0.1,
          11: 0.05,
          12: 0.05,
          13: 0.05,
          14: 0.05 }

knowledge_components = {1: "One digit addition without overflow",
                        2: "One digit addition with overflow",
                        3: "Two digit addition without carry and no overflow",
                        4: "Two digit addition without carry and overflow",
                        5: "Two digit addition with carry and no overflow",
                        6: "Two digit addition with carry and overflow",
                        7: "Three digit addition without carry and no overflow",
                        8: "Three digit addition without carry and overflow",
                        9: "Three digit addition with carry and no overflow",
                        10: "Three digit addition with carry and overflow",
                        11: "Four digit addition without carry and no overflow",
                        12: "Four digit addition without carry and overflow",
                        13: "Four digit addition with carry and no overflow",
                        14: "Four digit addition with carry and overflow"}

class LFAStudent2(object):

    def __init__(self,kc,lr,df):
        self.kcUseCount = {x:0 for x in kc}
        self.lr = lr
        self.diffFactor = df

    def computeProb(self,kcList):
        coeff = (self.lr/len(kcList))*sum([self.kcUseCount[kc]*lrKc[kc] for kc in kcList])
       # print(f"{alpha}--{kcList}--{coeff}")
        return (1/(1+pow(e,-coeff+self.diffFactor)))

    def getProficiency(self,kc):
        coeff = self.kcUseCount[kc]*lrKc[kc]*self.lr
        return (1/(1+pow(e,-coeff+self.diffFactor)))

    def solve(self, problem):
        kc = self.getKc(problem)
        preList = self.getPrerequisite(kc).copy()
        preList.append(kc)
        #print(preList)
        #probList = [self.getProficiency(x) for x in preList]
        #alpha = reduce(lambda x, y: x * y, probList, 1)
        #pre_reqs = self._get_prerequisites(kc)
        #print(f"problem: {problem} kc: {kc} state:{self.kc_states[kc]}")

        self.kcUseCount[kc] +=1

        #p_kcs = [(1 - self.ps[x]) if self.kc_states[x] == KC_STATE.LEARNED else self.pg[x] for x in pre_reqs]
        #p_kcs.append((1 - self.ps[kc]) if self.kc_states[kc] == KC_STATE.LEARNED else self.pg[kc])
        p_solving = self.computeProb(preList)
        #print(p_solving)

        solution = self.getSolution(problem) if random() < p_solving else -inf

        return solution

    def status(self):
        status = [f"\nState of KC {x[0]} is {x[1]}." for x in self.kc_states.items()]
        return ("").join(status)

    def getKc(self,problem):
        trace = problem[3]

        n_symbols_dict = {key:0 for key in ["A", "B", "C", "D"]}
        for t in trace:
            if t == "A":
                n_symbols_dict["A"] += 1
            elif t == "B":
                n_symbols_dict["B"] += 1
            elif t == "C":
                n_symbols_dict["C"] += 1
            elif t == "D":
                n_symbols_dict["D"] += 1

        # 14: "Four digit addition with carry and overflow" (A, B, D)
        if n_symbols_dict["A"] + n_symbols_dict["B"] == 4 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
            kc = 14
        # 13: "Four digit addition with carry and no overflow", (A, B)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 4 and n_symbols_dict["B"] > 0:
            kc = 13
        # 12: "Four digit addition without carry and overflow", (A, A, A, A, D)
        elif n_symbols_dict["A"] == 4 and n_symbols_dict["D"] > 0:
            kc = 12
        # 11: "Four digit addition without carry and no overflow", (A, A, A, A)
        elif n_symbols_dict["A"]== 4:
            kc = 11
        # 10: "Three digit addition with carry and overflow, (A, B, D)"
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 3 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
            kc = 10
        # 9: "Three digit addition with carry and no overflow", (A, B)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 3 and n_symbols_dict["B"] > 0:
            kc = 9
        # 8: "Three digit addition without carry and overflow", (A, A, A, D)
        elif n_symbols_dict["A"] == 3 and n_symbols_dict["D"] > 0:
            kc = 8
        # 7: "Three digit addition without carry and no overflow", (A, A, A)
        elif n_symbols_dict["A"] == 3:
            kc = 7
        # 6: "Two digit addition with carry and overflow", (A, B, D)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 2 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
            kc = 6
        # 5: "Three digit addition with carry and no overflow", (A, B)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 2 and n_symbols_dict["B"] > 0:
            kc = 5
        # 4: "Two digit addition without carry and overflow", (A, A, D)
        elif n_symbols_dict["A"] == 2 and n_symbols_dict["D"] > 0:
            kc = 4
        # 3: "Two digit addition without carry and no overflow", (A, A)
        elif n_symbols_dict["A"] == 2:
            kc = 3
        # 2: "One digit addition with overflow.", (A, C)
        elif n_symbols_dict["A"] == 1 and n_symbols_dict["C"] == 1:
            kc = 2
        # 1: "One digit addition without overflow.", (A)
        elif n_symbols_dict["A"] == 1:
            kc = 1

        # logger.info(f"KC: {kc}")

        return kc

    def getSolution(self,problem):
        return problem[2]

    def getPrerequisite(self,kc):
        return knowledge_component_prerequisites[kc]

### **PFA Student Model**

In [ ]:
kc = {1: "One digit addition without overflow",
      2: "One digit addition with overflow",
      3: "Two digit addition without carry",
      4: "Two digit addition with one carry",
      5: "Two digit addition without overflow",
      6: "Two digit addition with overflow",
      7: "Three digit addition without carry",
      8: "Three digit addition with one carry",
      9: "Three digit addition with two carry",
      10: "Three digit addition without overflow",
      11: "Three digit addition with overflow",
      12: "Four digit addition without carry",
      13: "Four digit addition with one carry",
      14: "Four digit addition with two carry",
      15: "Four digit addition with three carry",
      16: "Four digit addition without overflow",
      17: "Four digit addition with overflow" }

kc_prerequisite = {
    1 : [],
    2 : [1],
    3 : [1],
    4 : [2,3],
    5 : [1],
    6 : [2],
    7 : [3],
    8 : [7],
    9 : [8],
    10 : [5],
    11 : [6],
    12 : [7],
    13 : [12],
    14 : [13],
    15 : [14],
    16 : [10],
    17 : [11]
}

srKc = {
    1 : 0.1,
    2 : 0.1,
    3 : 0.1,
    4 : 0.00075,
    5 : 0.05,
    6 : 0.00075,
    7 : 0.075,
    8 : 0.05,
    9 : 0.05,
    10 : 0.075,
    11 : 0.05,
    12 : 0.05,
    13 : 0.025,
    14 : 0.025,
    15 : 0.025,
    16 : 0.025,
    17 : 0.025
}

frKc = {
    1 : 0.05,
    2 : 0.05,
    3 : 0.05,
    4 : 0.00025,
    5 : 0.025,
    6 : 0.0003,
    7 : 0.03,
    8 : 0.025,
    9 : 0.025,
    10 : 0.03,
    11 : 0.025,
    12 : 0.025,
    13 : 0.01,
    14 : 0.01,
    15 : 0.01,
    16 : 0.01,
    17 : 0.01
}

class PFAStudent(object):

    def __init__(self,kc,lr,df):
        self.kcSCount = {x:0 for x in kc}
        self.kcFCount = {x:0 for x in kc}
        self.lr = lr
        self.df = df

    def computeProb(self,kcList,alpha):
        coeff = (self.lr/len(kcList))*sum([(self.kcSCount[x]*srKc[x]*2.5 + self.kcFCount[x]*frKc[x]) for x in kcList]) *alpha
        return (1/(1+pow(e,-coeff+self.df)))

    def getProficiency(self,kc):
        coeff = (self.kcSCount[kc]*srKc[kc] + self.kcFCount[kc]*frKc[kc])*self.lr
        return (1/(1+pow(e,-coeff+self.df)))

    def solve(self, problem):
        kcList = self.getKc(problem)
        flatList = [self.getPrerequisite(x) for x in kcList]
        kcPreList = [item for sublist in flatList for item in sublist]
        probList = [self.getProficiency(x) for x in kcPreList]
        alpha = reduce(lambda x, y: x * y, probList, 1)
        #pre_reqs = self._get_prerequisites(kc)
        #print(f"problem: {problem} kc: {kc} state:{self.kc_states[kc]}")

        # for x in kcList:
        #   self.kcUseCount[x] += 1

        #p_kcs = [(1 - self.ps[x]) if self.kc_states[x] == KC_STATE.LEARNED else self.pg[x] for x in pre_reqs]
        #p_kcs.append((1 - self.ps[kc]) if self.kc_states[kc] == KC_STATE.LEARNED else self.pg[kc])
        p_solving = self.computeProb(kcList,alpha)
        #print(p_solving)

        if random() < p_solving:
          solution = self.getSolution(problem)
          for x in kcList:
            self.kcSCount[x] += 1
        else:
          solution = -inf
          for x in kcList:
            self.kcFCount[x] += 1
        return solution

    def status(self):
        status = [f"\nState of KC {x[0]} is {x[1]}." for x in self.kc_states.items()]
        return ("").join(status)

    def getKc(self,problem):
        trace = problem[3]
        kcList = []

        n_symbols_dict = {key:0 for key in ["A", "B", "C", "D"]}
        for t in trace:
            if t == "A":
                n_symbols_dict["A"] += 1
            elif t == "B":
                n_symbols_dict["B"] += 1
            elif t == "C":
                n_symbols_dict["C"] += 1
            elif t == "D":
                n_symbols_dict["D"] += 1


        #Four digit addition
        if n_symbols_dict["A"] + n_symbols_dict["B"] == 4:
          if n_symbols_dict["D"]:
            kcList.append(17)
          else:
            kcList.append(16)

          if n_symbols_dict["A"]==4:
            kcList.append(12)
          elif n_symbols_dict["A"]==3:
            kcList.append(13)
          elif n_symbols_dict["A"]==2:
            kcList.append(14)
          else:
            kcList.append(15)

        #Three digit addition
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 3:
          if n_symbols_dict["D"]:
            kcList.append(11)
          else:
            kcList.append(10)

          if n_symbols_dict["A"]==3:
            kcList.append(7)
          elif n_symbols_dict["A"]==2:
            kcList.append(8)
          else:
            kcList.append(9)

        #Two digit addition
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 2:
          if n_symbols_dict["D"]:
            kcList.append(6)
          else:
            kcList.append(5)

          if n_symbols_dict["A"]==2:
            kcList.append(3)
          else:
            kcList.append(4)

        #One digit addition
        else:
          if n_symbols_dict["D"]:
            kcList.append(2)
          else:
            kcList.append(1)

        # logger.info(f"KC: {kc}")

        return kcList

    def getSolution(self,problem):
        return problem[2]

    def getPrerequisite(self,kc):
        return kc_prerequisite[kc]


In [ ]:
knowledge_components = {1: "One digit addition without overflow",
                        2: "One digit addition with overflow",
                        3: "Two digit addition without carry and no overflow",
                        4: "Two digit addition without carry and overflow",
                        5: "Two digit addition with carry and no overflow",
                        6: "Two digit addition with carry and overflow",
                        7: "Three digit addition without carry and no overflow",
                        8: "Three digit addition without carry and overflow",
                        9: "Three digit addition with carry and no overflow",
                        10: "Three digit addition with carry and overflow",
                        11: "Four digit addition without carry and no overflow",
                        12: "Four digit addition without carry and overflow",
                        13: "Four digit addition with carry and no overflow",
                        14: "Four digit addition with carry and overflow"}

srKc = { 1: 0.2,
        2: 0.2,
        3: 0.15,
        4: 0.15,
        5: 0.15,
        6: 0.15,
        7: 0.1,
        8: 0.1,
        9: 0.1,
        10: 0.1,
        11: 0.05,
        12: 0.05,
        13: 0.05,
        14: 0.05 }

frKc = {
    1 : 0.05,
    2 : 0.05,
    3 : 0.03,
    4 : 0.03,
    5 : 0.03,
    6 : 0.03,
    7 : 0.02,
    8 : 0.02,
    9 : 0.02,
    10 : 0.02,
    11 : 0.01,
    12 : 0.01,
    13 : 0.01,
    14 : 0.01
}

class PFAStudent1(object):

    def __init__(self,kc,lr,df):
        self.kcSCount = {x:0 for x in kc}
        self.kcFCount = {x:0 for x in kc}
        self.lr = lr
        self.df = df

    def computeProb(self,kcList):
        coeff = (self.lr/len(kcList))*sum([(self.kcSCount[x]*srKc[x] + self.kcFCount[x]*frKc[x]) for x in kcList])
        return (1/(1+pow(e,-coeff+self.df)))

    def getProficiency(self,kc):
        coeff = (self.kcSCount[kc]*srKc[kc] + self.kcFCount[kc]*frKc[kc])*self.lr
        return (1/(1+pow(e,-coeff+self.df)))

    def solve(self, problem):
        kc = self.getKc(problem)
        preList = self.getPrerequisite(kc).copy()
        preList.append(kc)
        #pre_reqs = self._get_prerequisites(kc)
        #print(f"problem: {problem} kc: {kc} state:{self.kc_states[kc]}")

        # for x in kcList:
        #   self.kcUseCount[x] += 1

        #p_kcs = [(1 - self.ps[x]) if self.kc_states[x] == KC_STATE.LEARNED else self.pg[x] for x in pre_reqs]
        #p_kcs.append((1 - self.ps[kc]) if self.kc_states[kc] == KC_STATE.LEARNED else self.pg[kc])
        p_solving = self.computeProb(preList)
        #print(p_solving)

        if random() < p_solving:
          solution = self.getSolution(problem)
          self.kcSCount[kc] += 1
        else:
          solution = -inf
          self.kcFCount[kc] += 1
        return solution

    def getKc(self,problem):
        trace = problem[3]

        n_symbols_dict = {key:0 for key in ["A", "B", "C", "D"]}
        for t in trace:
            if t == "A":
                n_symbols_dict["A"] += 1
            elif t == "B":
                n_symbols_dict["B"] += 1
            elif t == "C":
                n_symbols_dict["C"] += 1
            elif t == "D":
                n_symbols_dict["D"] += 1

        # 14: "Four digit addition with carry and overflow" (A, B, D)
        if n_symbols_dict["A"] + n_symbols_dict["B"] == 4 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
            kc = 14
        # 13: "Four digit addition with carry and no overflow", (A, B)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 4 and n_symbols_dict["B"] > 0:
            kc = 13
        # 12: "Four digit addition without carry and overflow", (A, A, A, A, D)
        elif n_symbols_dict["A"] == 4 and n_symbols_dict["D"] > 0:
            kc = 12
        # 11: "Four digit addition without carry and no overflow", (A, A, A, A)
        elif n_symbols_dict["A"]== 4:
            kc = 11
        # 10: "Three digit addition with carry and overflow, (A, B, D)"
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 3 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
            kc = 10
        # 9: "Three digit addition with carry and no overflow", (A, B)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 3 and n_symbols_dict["B"] > 0:
            kc = 9
        # 8: "Three digit addition without carry and overflow", (A, A, A, D)
        elif n_symbols_dict["A"] == 3 and n_symbols_dict["D"] > 0:
            kc = 8
        # 7: "Three digit addition without carry and no overflow", (A, A, A)
        elif n_symbols_dict["A"] == 3:
            kc = 7
        # 6: "Two digit addition with carry and overflow", (A, B, D)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 2 and n_symbols_dict["B"] > 0 and n_symbols_dict["D"] > 0:
            kc = 6
        # 5: "Three digit addition with carry and no overflow", (A, B)
        elif n_symbols_dict["A"] + n_symbols_dict["B"] == 2 and n_symbols_dict["B"] > 0:
            kc = 5
        # 4: "Two digit addition without carry and overflow", (A, A, D)
        elif n_symbols_dict["A"] == 2 and n_symbols_dict["D"] > 0:
            kc = 4
        # 3: "Two digit addition without carry and no overflow", (A, A)
        elif n_symbols_dict["A"] == 2:
            kc = 3
        # 2: "One digit addition with overflow.", (A, C)
        elif n_symbols_dict["A"] == 1 and n_symbols_dict["C"] == 1:
            kc = 2
        # 1: "One digit addition without overflow.", (A)
        elif n_symbols_dict["A"] == 1:
            kc = 1

        # logger.info(f"KC: {kc}")

        return kc

    def getSolution(self,problem):
        return problem[2]

    def getPrerequisite(self,kc):
        return knowledge_component_prerequisites[kc]


##**Tutoring Algorithm**

In [ ]:
class KLUCB_Node2(object):
    """
    Class for objects corresponding to the nodes in the UCB algorithm
    """
    def __init__(self,t,a1,b1,a2,b2):
        # these values are for MAB
        self.timesPlayed = 0
        self.correctnessSum = 0
        self.estimate = 0
        self.ucb = 0
        self.lcb = 0
        self.timeAdded = t

        # these values are for CUSUM
        self.correctnessRecord = []
        self.log_seq = []
        self.pg =  np.random.beta(a1,b1)
        self.ps = np.random.beta(a2,b2)
        self.cdEstimate = 0
        self.index = 0

    def update_estimate(self, correctness):
        self.timesPlayed += 1
        self.correctnessSum += correctness
        self.estimate = round(self.correctnessSum/self.timesPlayed,3)
        self.correctnessRecord.append(correctness)

    def compute_lcb_ucb(self, t):
        val = round(log(1 + (t)*pow(log(t),2))/self.timesPlayed,3)
        #print(f"val:{val}, muE:{self.estimate}")
        self.ucb = round(getUCB(val,self.estimate),3)
        self.lcb = round(getLCB(val,self.estimate),3)
        #print(f"timeAdded: {self.timeAdded} val:{val}, muE:{self.estimate}, UCB:{self.ucb}, LCB:{self.lcb}")

    def cd(self,h):
        est = log((1-self.ps)/self.pg) if self.correctnessRecord[-1] else log(self.ps/(1-self.pg))
        if est>0 and self.cdEstimate==0:
          self.index = self.timesPlayed
        self.cdEstimate = self.cdEstimate+est
        if self.cdEstimate<0:
          self.cdEstimate = 0
          self.index = 0
        self.log_seq.append(self.cdEstimate)
        print(f"History of correctness of responses of the chosen concept: {self.correctnessRecord}")

        if self.cdEstimate>=h:
          return True
        return False

    def __str__(self):
        return (f"timesPlayed: {self.timesPlayed}, estimate: "
                f"{round(self.estimate,2)}, ucb: {round(self.ucb,2)},"
                f"lcb: {round(self.lcb,2)}, timeAdded: {self.timeAdded},"
                f" index: {self.index}, pg: {round(self.pg,2)}, ps: "
                f"{round(self.ps,2)}, correctnessRecord: {self.correctnessRecord}, \n")

def getUCB(value,est):
  if kl_div(est,0.999)<value:
    return 1
  while True:
    try:
      res = optimize.newton(kl(est,value),fprime=klPrime(est),fprime2=klDPrime(est),x0=0.99+abs(np.random.normal(0.0001,0.01)))
      break
    except ValueError:
      pass
  return res

def getLCB(value,est):
  if kl_div(est,0.001)<value:
    return 0
  while True:
      try:
        res = optimize.newton(kl(est,value),fprime=klPrime(est),fprime2=klDPrime(est),x0=0.01-abs(np.random.normal(0.0001,0.01)))
        break
      except ValueError:
        pass
  return res

def is_correct(problem, answer):
    return answer == problem[2]

def kl_div(x,y):
  if x==y:
    return 0
  if x==1:
    return x*log(x/y)
  if y==1:
    return inf
  if x>0 and y>0:
    return (x*log(x/y))+(1-x)*log((1-x)/(1-y))
  if x>0:
    return inf
  return log(1/(1-y))

def kl(p,val):
  if p==1:
    return lambda x: log(1/x) - val
  elif p==0:
    return lambda x: log(1/(1-x)) - val
  pTerm = (p*log(p) + (1-p)*log(1-p))
  return lambda x: pTerm - p*log(x) - (1-p)*log(1-x) - val

def klPrime(p):
  if p==0:
    return lambda x: 1/(1-x)
  return lambda x:(-p/x) + ((1-p)/(1-x))

def klDPrime(p):
  return lambda x:(-p/pow(x,2)) + ((1-p)/pow(1-x,2))

def klucbCUSUM(init_zpd,
          trace_problems_dict,
          progression_graph,
          fpr,
          a1,b1,
          a2,b2,
          student):
    """
    Function that runs the vanilla ucb1 algorithm on the given student according
    to the given parameters.

    Parameter(s):
        init_zpd: set(string)
        trace_problems_dict: dict(string -> list(tuple(int, int, int, string)))
        progression_graph: dict(string -> list(string))
        fpr:float
        a1: int
        b1: int
        a2: int
        b2: int
        student: Student

    """
    # A map from traces to iterators of the corresponding problem list.
    trace_problemsIter_dict = {}
    for trace, problems in trace_problems_dict.items():
        trace_problemsIter_dict[trace] = iter(problems)

    # A map from all traces in ZPD to a corresponding UCB1_node object.
    ucb1_trace_node_dict = {}

    for trace in init_zpd:
        ucb1_trace_node_dict[trace] = KLUCB_Node2(0,a1,b1,a2,b2)

    # Generate list of all unsolvable traces
    unsolvable = set()
    solvable = set()
    seedSet = init_zpd.copy()
    while len(seedSet) > 0:
      trace = seedSet.pop()
      unsolvable.update(progression_graph[trace])
      seedSet.update(progression_graph[trace])

    unsolvable = unsolvable - init_zpd

    # A map from unsolvable traces to their ancestors
    trace_ancestor_dict = defaultdict(list)
    seedSet = init_zpd.copy()
    while len(seedSet)>0:
      trace = seedSet.pop()
      for t in progression_graph[trace]:
        if t in unsolvable:
          trace_ancestor_dict[t].append(trace)
        seedSet.update(progression_graph[trace])

    ts = 0
    beta = log(1/fpr)
    solvableRatio = []
    zpd = list(init_zpd)


    while len(ucb1_trace_node_dict) > 0:
        ts = ts+1
        flag = False
        print(f"\nQuestion: {ts}")

        #Check if there are any unplayed arms in the ZPD
        for trace,node in ucb1_trace_node_dict.items():
          if node.timesPlayed==0:
            chosen_trace = trace
            flag = True
            break

        #Selecting the arm(problem) with the highest ucb when all arms have been
        #played atleast once

        if flag is False:

            # Selecting the problem with the highest ucb
            trace_ucb_dict = {}

            #Computing the n_t value for CD policy
            nt = 0
            for trace, node in ucb1_trace_node_dict.items():
              nt += node.timesPlayed

            for trace, node in ucb1_trace_node_dict.items():
              node.compute_lcb_ucb(nt)
              trace_ucb_dict[trace] = node.ucb

            chosen_trace = max(trace_ucb_dict,key=lambda key:trace_ucb_dict[key])

        # Select a problem from the problem list of the chosen_trace consecutively
        # and loop around the list when exhausted.
        try:
            problem = next(trace_problemsIter_dict[chosen_trace])
        except StopIteration:
            trace_problemsIter_dict[chosen_trace] = iter(trace_problems_dict[chosen_trace])
            problem = next(trace_problemsIter_dict[chosen_trace])

        print(f"Chosen concept: {chosen_trace}")
        print(f"Selected problem: {problem}")

        # Update the reward estimates from the arm based on student response
        answer = student.solve(problem)
        correctness = is_correct(problem, answer)
        chosen_node = ucb1_trace_node_dict[chosen_trace]
        chosen_node.update_estimate(correctness)
        #print(f"{chosen_trace}---{trace_ucb_dict[chosen_trace]}--{correctness}")
        print(f"Correctness of student response: {correctness}")

        # Check if a change point has occurred using CUSUM
        if chosen_node.cd(beta):
            # Update ZPD.
            print(f"\nMastery detected for {chosen_trace}!!")
            unsolvable.discard(chosen_trace)
            solvable.add(chosen_trace)
            zpd.remove(chosen_trace)

            for t in progression_graph[chosen_trace]:
                if t in trace_ancestor_dict and any([tr in unsolvable for tr in trace_ancestor_dict[t]]):
                  continue
                if t not in ucb1_trace_node_dict:
                  ucb1_trace_node_dict[t] = KLUCB_Node2(ts,a1,b1,a2,b2)
                  zpd.append(t)

                  #print(f"Added {t}")
                  #l.append(t)
            #print("------------------------------")
            del ucb1_trace_node_dict[chosen_trace]
            #l.remove(chosen_trace)
            # del trace_problemIters_dict[chosen_trace]

        #solvableRatio.append(len(solvable)/(len(solvable)+len(unsolvable)))

    return

In [ ]:
student = Student(student1_kc_states,
                  transition_probabilities,
                  slip_probabilities,
                  guess_probabilities,
                  get_prerequisites,
                  get_kc,
                  get_solution)

klucbCUSUM({"A"},trace_problems_dict,progression_graph,0.00009,20,160,20,160,student)

In [ ]:
student = LFAStudent2(kc,1,0.8)

klucbCUSUM({"A"},trace_problems_dict,progression_graph,0.0004,20,160,20,160,student)